In [12]:
import torch 
import math
import numpy as np 

In [48]:
# Understanding attention mechanism 
# Scaled Dot-Product Attention 
# Using Q , K , V 
def scaled_dot_product_attention(Q,K,V,mask=True):
    #1 Multiply
    R = Q @ K.T
    #2 Scale
    dk = math.sqrt(K.shape[1])
    R = R/dk
    #3 Mask optional 
    if mask:
        mask_ = torch.tril(torch.ones(Q.shape[0], Q.shape[0]))
        R = R.masked_fill(mask_ ==0 , -float('inf'))
    #4 Softmax 
    S = torch.softmax(R,dim=-1)
    #5 Mutiply 
    F = S @ V
    # print("Scaled Dot Product Attention : ")
    # print(F)
    return F

sequence_len = 3 
Q = torch.randn(sequence_len, 4)
K = torch.randn(sequence_len, 4)
V = torch.randn(sequence_len, 5)
scaled_dot_product_attention(Q,K,V)



tensor([[-0.4172,  0.3057,  1.0319, -0.0998,  1.3516],
        [-0.0275,  0.4175,  0.7726, -0.3411,  1.4015],
        [ 0.2839,  0.5010,  1.0190, -0.5445,  0.8790]])

In [50]:
# Understanding attention mechanism 
# Implementing multihead attention 
# Using Q , K , V 


def multi_head_attention(Q,K,V,heads=1):

    assert Q.shape[1] % heads == 0
    assert K.shape[1] % heads == 0
    assert V.shape[1] % heads == 0

    feat = K.shape[1]

    dh = feat//heads 
    dv = V.shape[1]//heads

    

    Q_s = ([Q[:,i:i+dh] for i in range(0,feat,dh)])
    K_s = ([K[:,i:i+dh] for i in range(0,feat,dh)])
    V_s = ([V[:,i:i+dv] for i in range(0,V.shape[1],dv)])



    N = len(Q_s)

    HEADS_OP = []

    for i in range(N):

        WQ = torch.rand(dh, dh)
        WK = torch.rand(dh, dh)
        WV = torch.rand(dv, dv)

        Q_h = Q_s[i] @ WQ.T
        K_h = K_s[i] @ WK.T
        V_h = V_s[i] @ WV.T


        output = scaled_dot_product_attention(
            Q_h, K_h, V_h, mask=None
        )
        HEADS_OP.append(output)
    print(HEADS_OP)
    total = torch.cat(HEADS_OP, dim=-1)
    print(total)
    return total 

sequence_len = 3 
Q = torch.randn(sequence_len, 4)
K = torch.randn(sequence_len, 4)
V = torch.randn(sequence_len, 4)

multi_head_attention(Q,K,V,heads=2)


[tensor([[-0.4173, -0.1460],
        [-0.3099, -0.1080],
        [-0.4741, -0.1707]]), tensor([[-0.2321, -0.4156],
        [-0.2166, -0.3877],
        [-0.1923, -0.3407]])]
tensor([[-0.4173, -0.1460, -0.2321, -0.4156],
        [-0.3099, -0.1080, -0.2166, -0.3877],
        [-0.4741, -0.1707, -0.1923, -0.3407]])


tensor([[-0.4173, -0.1460, -0.2321, -0.4156],
        [-0.3099, -0.1080, -0.2166, -0.3877],
        [-0.4741, -0.1707, -0.1923, -0.3407]])